# Installation
`sagea==0.3.2` and later versions support installation in different environments, including standard Python (>=3.12) environments and lightweight installation in JupyterLite environments. In web-based environments such as JupyterLite, some features are currently unavailable due to limited support for certain dependencies. This setup is intended for demonstration purposes only. For full functionality, please install and use sagea in a standard Python environment. Run the example notebook in a new tab.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%pip install sagea==0.3.2

"""
When installing sagea in web-based environments such as JupyterLite, only a minimal set of basic dependencies related to numerical computation is included by default. Therefore, additional plotting libraries or related dependencies need to be installed manually for later use.
"""
%pip install matplotlib
%pip install cartopy
%pip install geopandas
%pip install rasterio
%pip install affine

# Load gravity products (SHCs)

In [ ]:
import sagea
from sagea.utils import TimeTool
from sagea.sgio import read_low_degs
import pathlib

# define paths of products
pathlist_l2 = list(pathlib.Path("./data/GRACE_L2_GSM_Products/").glob("GSM-2_*_UTCSR_BA01_0600"))
pathlist_l2.sort()

path_gif48 = pathlib.Path("./data/auxiliary/GIF48.60.gfc")

path_TN14 = pathlib.Path("./data/GRACE_L2_Low_Degree_Products/TN-14_C30_C20_SLR_GSFC.txt")

path_GIA = pathlib.Path("./data/GIA/GIA.ICE-6G_D.txt")

# load products as SHC instance, match dates information
lmax = 60

shc = sagea.SHC.io.from_gfc(pathlist_l2, lmax=lmax, key="GRCOF2")
shc_gif48 = sagea.SHC.io.from_gfc(path_gif48, lmax=lmax, key="gfc")
shc_gia_trend = sagea.SHC.io.from_gfc(path_GIA, lmax=lmax, key="")

dates_begin, dates_end = TimeTool.match_dates_from_name(pathlist_l2)
dates = TimeTool.get_average_dates(dates_begin, dates_end)

# read low-degree product
dict_low_degs = read_low_degs(path_TN14, dates)
c20 = dict_low_degs["c2,0"]


# Post-processing
Due to limitations in providing large amounts of data in the web-based environment, only a few steps in the post-processing are demonstrated here.

In [ ]:
# replace c20 coefficients
shc.replace("c2,0", c20, inplace=True)

# deduct background
shc -= shc_gif48

# deduct GIA singal
shc_gia = sagea.SHC.generate.from_trend(shc_gia_trend, dates=dates)
shc -= shc_gia

# set C(0,0) values as zero, which may be set as different constants (e.g., 1 or 0) in some products
shc.replace("c0,0", 0, inplace=True)

# filtering
shc_filtered = shc.filter.slidewindowSwenson2006(n=3, m=5, a=30, k=10, window_length_min=5)
shc_filtered.filter.gaussian(radius=300, inplace=True)

# convert shc's physical dimension into EWH
shc_ewh_unfiltered = shc.convert(from_type='Geopotential', to_type='EWH')  # EWH in unit [m]
shc_ewh_filtered = shc_filtered.convert(from_type='Geopotential', to_type='EWH')


# Synthesis into gridded EWH field and show the spatial distribution

In [ ]:
import cartopy

# Since the web-based JupyterLite environment cannot access online download services, the necessary local terrain/coastline datasets for plotting are provided separately and configured explicitly in this section.
cartopy.config["data_dir"] = "./data/cartopy_data"

# spherical harmonic synthesis into EWH grid
grid_space = 1  # in unit [degree]
grid_unfiltered = shc_ewh_unfiltered.synthesize.to_grid(grid_space=grid_space)
grid_filtered = shc_ewh_filtered.synthesize.to_grid(grid_space=grid_space)

for grid in [grid_unfiltered, grid_filtered]:
    grid.plot(
        index=3,
        vmin=-0.3, vmax=0.3,
        projection=cartopy.crs.Robinson(),
        gridlines=False,
        coastline=True
    )

# Extract basin-scale signals

In [ ]:
from sagea.sgio import read_shp_as_GRD

grid_space = 1
filepath_mask = pathlib.Path("./data/shpfiles/Dunube_subbasins_3")
grid_mask = read_shp_as_GRD(filepath_mask, grid_space=grid_space, per_feature=True)

# show the masks
fig, axes = grid_mask.plot(
    title="Masks",
    gridlines=True,
    vmin=0, vmax=1,
    projection=cartopy.crs.Robinson(),
    extent=(5, 35, 40, 53),
    cmap="Blues",
)

# extract basin signal, plot time series
ewh_series = grid.extract.maskGRD(grid_mask, average=True)  # in shape (n_mask, n_time)
year_fraction = TimeTool.convert_date_format(dates, input_type=TimeTool.DateFormat.ClassDate,
                                             output_type=TimeTool.DateFormat.YearFraction)

for i in range(ewh_series.shape[0]):
    plt.plot(year_fraction, ewh_series[i] * 100, label=f"subbasin index{i}")  # plot in unie [cm]

plt.xticks(np.arange(2008, 2009, 1 / 12),
           ["Jan", "", "Mar", "", "May", "", "Jul", "", "Sep", "", "Nov", ""])

plt.grid(ls="--", color="k", alpha=0.3)
plt.xlabel("Time")
plt.ylabel("EWH (cm)")
plt.legend()
plt.show()